<a href="https://colab.research.google.com/github/prajwalmotagi/imdb-movie-recommender/blob/main/imdb_movie_recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IMDb Movie Recommender
Built a weighted rating system to fix vote-count bias in IMDb ratings.
Loads 5,000 real movies, calculates fair scores using Bayesian-style
weighting, and ranks movies by trustworthiness of their rating.

In [23]:
movies = [
    {"name": "Tiny Indie Film", "rating": 9.5, "votes": 200},
    {"name": "The Shawshank Redemption", "rating": 9.3, "votes": 2800000},
    {"name": "Random Short Film", "rating": 9.6, "votes": 45},
    {"name": "The Dark Knight", "rating": 9.0, "votes": 2700000},
    {"name": "The Outsider", "rating": 6.3, "votes": 31746},
    {"name": "The Kashmir Files", "rating": 8.5, "votes": 579183},
    {"name": "Parasite", "rating": 8.5, "votes": 1171802}
]

print(movies)

[{'name': 'Tiny Indie Film', 'rating': 9.5, 'votes': 200}, {'name': 'The Shawshank Redemption', 'rating': 9.3, 'votes': 2800000}, {'name': 'Random Short Film', 'rating': 9.6, 'votes': 45}, {'name': 'The Dark Knight', 'rating': 9.0, 'votes': 2700000}, {'name': 'The Outsider', 'rating': 6.3, 'votes': 31746}, {'name': 'The Kashmir Files', 'rating': 8.5, 'votes': 579183}, {'name': 'Parasite', 'rating': 8.5, 'votes': 1171802}]


In [24]:
C = 7.0      # the "default expected rating" every movie starts near
m = 25000    # how many votes it takes before we mostly trust its own rating

def weighted_rating(votes, rating):
    return (votes / (votes + m)) * rating + (m / (votes + m)) * C

print(weighted_rating(200, 9.5))
print(weighted_rating(2800000, 9.3))

7.01984126984127
9.279646017699115


In [25]:
for movie in movies:
    movie["weighted_score"] = weighted_rating(movie["votes"], movie["rating"])

for movie in movies:
    print(movie["name"], "→", round(movie["weighted_score"], 2))

Tiny Indie Film → 7.02
The Shawshank Redemption → 9.28
Random Short Film → 7.0
The Dark Knight → 8.98
The Outsider → 6.61
The Kashmir Files → 8.44
Parasite → 8.47


In [26]:
ranked = sorted(movies, key=lambda m: m["weighted_score"], reverse=True)

print("--- FINAL RANKING ---")
for i, movie in enumerate(ranked):
    print(f"{i+1}. {movie['name']} | rating: {movie['rating']} | weighted: {round(movie['weighted_score'], 2)}")

--- FINAL RANKING ---
1. The Shawshank Redemption | rating: 9.3 | weighted: 9.28
2. The Dark Knight | rating: 9.0 | weighted: 8.98
3. Parasite | rating: 8.5 | weighted: 8.47
4. The Kashmir Files | rating: 8.5 | weighted: 8.44
5. Tiny Indie Film | rating: 9.5 | weighted: 7.02
6. Random Short Film | rating: 9.6 | weighted: 7.0
7. The Outsider | rating: 6.3 | weighted: 6.61


In [27]:
def recommend(movie_list, top_n=3):
    for movie in movie_list:
        movie["weighted_score"] = weighted_rating(
            movie["votes"], movie["rating"]
        )
    ranked = sorted(
        movie_list,
        key=lambda m: m["weighted_score"],
        reverse=True
    )
    print(f"--- TOP {top_n} RECOMMENDATIONS ---")
    for i, movie in enumerate(ranked[:top_n]):
        print(f"{i+1}. {movie['name']} | weighted: {round(movie['weighted_score'], 2)}")

recommend(movies, top_n=3)

--- TOP 3 RECOMMENDATIONS ---
1. The Shawshank Redemption | weighted: 9.28
2. The Dark Knight | weighted: 8.98
3. Parasite | weighted: 8.47


In [28]:
import pandas as pd

df = pd.read_csv("results_with_crew.csv")

print(df.shape)
print(df.head())

(5000, 12)
      tconst                                   primaryTitle  startYear  rank  \
0  tt0111161                       The Shawshank Redemption       1994     1   
1  tt0068646                                  The Godfather       1972     2   
2  tt0468569                                The Dark Knight       2008     3   
3  tt0167260  The Lord of the Rings: The Return of the King       2003     4   
4  tt0108052                               Schindler's List       1993     5   

   averageRating  numVotes  runtimeMinutes             directors  \
0            9.3   3198304             142        Frank Darabont   
1            9.2   2231582             175  Francis Ford Coppola   
2            9.1   3179134             152     Christopher Nolan   
3            9.0   2169878             201         Peter Jackson   
4            9.0   1588096             195      Steven Spielberg   

                                             writers  \
0                       Frank Darabont, Ste

In [29]:
C = df["averageRating"].mean()
m = df["numVotes"].quantile(0.25)

print("Average rating across all movies:", round(C, 2))
print("Minimum votes threshold:", round(m, 2))

Average rating across all movies: 7.18
Minimum votes threshold: 40005.0


In [30]:
def weighted_rating_df(row):
    v = row["numVotes"]
    R = row["averageRating"]
    return (v / (v + m)) * R + (m / (v + m)) * C

df["weighted_score"] = df.apply(weighted_rating_df, axis=1)

top_movies = df.sort_values("weighted_score", ascending=False).head(10)

print(top_movies[["primaryTitle", "averageRating", "numVotes", "weighted_score"]].to_string(index=False))

                                     primaryTitle  averageRating  numVotes  weighted_score
                         The Shawshank Redemption            9.3   3198304        9.273768
                                    The Godfather            9.2   2231582        9.164366
                                  The Dark Knight            9.1   3179134        9.076098
    The Lord of the Rings: The Return of the King            9.0   2169878        8.966992
                                 Schindler's List            9.0   1588096        8.955197
                            The Godfather Part II            9.0   1497791        8.952566
                                     12 Angry Men            9.0    987283        8.928993
The Lord of the Rings: The Fellowship of the Ring            8.9   2212021        8.869386
                                        Inception            8.8   2826603        8.777345
                                       Fight Club            8.8   2618882        8.775575

In [31]:
print(df["genres"].head(20))

0                           Drama
1                    Crime, Drama
2                 Crime, Thriller
3       Adventure, Drama, Fantasy
4       Biography, Drama, History
5                    Crime, Drama
6                    Crime, Drama
7       Adventure, Drama, Fantasy
8     Adventure, Sci-Fi, Thriller
9          Crime, Drama, Thriller
10                 Drama, Romance
11                   Crime, Drama
12      Adventure, Drama, Fantasy
13      Adventure, Drama, Western
14       Adventure, Drama, Sci-Fi
15                 Action, Sci-Fi
16     Adventure, Fantasy, Sci-Fi
17        Biography, Crime, Drama
18          Crime, Drama, Mystery
19           Crime, Drama, Horror
Name: genres, dtype: object


In [32]:
genres_encoded = df["genres"].str.get_dummies(sep=", ")

print("Total unique genres found:", genres_encoded.shape[1])
print(genres_encoded.head())

Total unique genres found: 23
   Action  Adventure  Animation  Biography  Comedy  Crime  Documentary  Drama  \
0       0          0          0          0       0      0            0      1   
1       0          0          0          0       0      1            0      1   
2       0          0          0          0       0      1            0      0   
3       0          1          0          0       0      0            0      1   
4       0          0          0          1       0      0            0      1   

   Family  Fantasy  ...  Music  Musical  Mystery  News  Romance  Sci-Fi  \
0       0        0  ...      0        0        0     0        0       0   
1       0        0  ...      0        0        0     0        0       0   
2       0        0  ...      0        0        0     0        0       0   
3       0        1  ...      0        0        0     0        0       0   
4       0        0  ...      0        0        0     0        0       0   

   Sport  Thriller  War  Western

In [33]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(genres_encoded)

print("Similarity matrix shape:", similarity_matrix.shape)
print("Similarity between movie 0 and movie 1:", round(similarity_matrix[0][1], 3))
print("Similarity between movie 0 and movie 2:", round(similarity_matrix[0][2], 3))

Similarity matrix shape: (5000, 5000)
Similarity between movie 0 and movie 1: 0.707
Similarity between movie 0 and movie 2: 0.0


In [34]:
def get_recommendations(title, n=5):
    # find the index of the movie in our dataset
    matches = df[df["primaryTitle"] == title]
    if matches.empty:
        print(f"Movie '{title}' not found.")
        return

    idx = matches.index[0]

    # get similarity scores for this movie against all others
    sim_scores = list(enumerate(similarity_matrix[idx]))

    # sort by similarity score
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # remove the movie itself (always has similarity 1.0 with itself)
    sim_scores = [s for s in sim_scores if s[0] != idx]

    # take top 20 similar movies
    top_indices = [s[0] for s in sim_scores[:20]]

    # from those 20, sort by weighted score to get best rated ones
    recommendations = df.iloc[top_indices][["primaryTitle", "averageRating", "numVotes", "weighted_score"]]
    recommendations = recommendations.sort_values("weighted_score", ascending=False).head(n)

    print(f"\nBecause you watched: {title}")
    print(f"We recommend:\n")
    print(recommendations[["primaryTitle", "averageRating", "weighted_score"]].to_string(index=False))

get_recommendations("Inception")
get_recommendations("The Godfather")
get_recommendations("The Dark Knight")
get_recommendations("Parasite")


Because you watched: Inception
We recommend:

                                  primaryTitle  averageRating  weighted_score
                                  Interstellar            8.7        8.676430
Star Wars: Episode V - The Empire Strikes Back            8.7        8.660645
                    Terminator 2: Judgment Day            8.6        8.557082
                            Back to the Future            8.5        8.464640
                             Avengers: Endgame            8.4        8.367239

Because you watched: The Godfather
We recommend:

         primaryTitle  averageRating  weighted_score
The Godfather Part II            9.0        8.952566
         12 Angry Men            9.0        8.928993
         Pulp Fiction            8.8        8.773828
          City of God            8.6        8.537640
   American History X            8.5        8.459768

Because you watched: The Dark Knight
We recommend:

         primaryTitle  averageRating  weighted_score
          

In [35]:
print(df["startYear"].min(), df["startYear"].max())

1920 2026


In [36]:
min_year = df["startYear"].min()
max_year = df["startYear"].max()

df["year_scaled"] = (df["startYear"] - min_year) / (max_year - min_year)

print("Min year:", min_year, "→ scaled:", df["year_scaled"].min())
print("Max year:", max_year, "→ scaled:", df["year_scaled"].max())
print("Year 1994 scaled:", round(df[df["startYear"] == 1994]["year_scaled"].values[0], 3))

Min year: 1920 → scaled: 0.0
Max year: 2026 → scaled: 1.0
Year 1994 scaled: 0.698


In [44]:
import numpy as np

year_weight = 1

combined_features = np.hstack([
    genres_encoded.values,
    df["year_scaled"].values.reshape(-1, 1) * year_weight
])

print("Combined feature matrix shape:", combined_features.shape)

Combined feature matrix shape: (5000, 24)


In [45]:
similarity_matrix_v2 = cosine_similarity(combined_features)

print("New similarity matrix shape:", similarity_matrix_v2.shape)

New similarity matrix shape: (5000, 5000)


In [47]:
def get_recommendations_v2(title, n=5, use_year=True):
    matches = df[df["primaryTitle"] == title]
    if matches.empty:
        print(f"Movie '{title}' not found.")
        return

    idx = matches.index[0]

    # choose which similarity matrix to use
    matrix = similarity_matrix_v2 if use_year else similarity_matrix

    sim_scores = list(enumerate(matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [s for s in sim_scores if s[0] != idx]

    top_indices = [s[0] for s in sim_scores[:20]]

    recommendations = df.iloc[top_indices][["primaryTitle", "startYear", "averageRating", "weighted_score"]]
    recommendations = recommendations.sort_values("weighted_score", ascending=False).head(n)

    print(f"\nBecause you watched: {title} ({int(df.iloc[idx]['startYear'])})")
    print(f"Mode: {'Genre + Year' if use_year else 'Genre only'}\n")
    print(recommendations[["primaryTitle", "startYear", "averageRating", "weighted_score"]].to_string(index=False))

# compare both versions for Inception
print("=" * 60)
get_recommendations_v2("Inception", use_year=False)
print("=" * 60)
get_recommendations_v2("Inception", use_year=True)


Because you watched: Inception (2010)
Mode: Genre only

                                  primaryTitle  startYear  averageRating  weighted_score
                                  Interstellar       2014            8.7        8.676430
Star Wars: Episode V - The Empire Strikes Back       1980            8.7        8.660645
                    Terminator 2: Judgment Day       1991            8.6        8.557082
                            Back to the Future       1985            8.5        8.464640
                             Avengers: Endgame       2019            8.4        8.367239

Because you watched: Inception (2010)
Mode: Genre + Year

           primaryTitle  startYear  averageRating  weighted_score
  2001: A Space Odyssey       1968            8.3        8.245531
          Jurassic Park       1993            8.2        8.166501
     Planet of the Apes       1968            8.0        7.868233
         Manjummel Boys       2024            8.2        7.646685
Furiosa: A Mad Max S

In [48]:
def get_recommendations_v3(title, n=5, era=None):
    # era options: "classic" (before 1980), "80s-90s" (1980-1999),
    #              "2000s" (2000-2009), "modern" (2010+)

    matches = df[df["primaryTitle"] == title]
    if matches.empty:
        print(f"Movie '{title}' not found.")
        return

    idx = matches.index[0]

    sim_scores = list(enumerate(similarity_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [s for s in sim_scores if s[0] != idx]

    top_indices = [s[0] for s in sim_scores[:50]]
    recommendations = df.iloc[top_indices].copy()

    # apply era filter if specified
    if era == "classic":
        recommendations = recommendations[recommendations["startYear"] < 1980]
    elif era == "80s-90s":
        recommendations = recommendations[
            (recommendations["startYear"] >= 1980) &
            (recommendations["startYear"] < 2000)
        ]
    elif era == "2000s":
        recommendations = recommendations[
            (recommendations["startYear"] >= 2000) &
            (recommendations["startYear"] < 2010)
        ]
    elif era == "modern":
        recommendations = recommendations[recommendations["startYear"] >= 2010]

    recommendations = recommendations.sort_values(
        "weighted_score", ascending=False
    ).head(n)

    era_label = era if era else "any era"
    print(f"\nBecause you watched: {title}")
    print(f"Era preference: {era_label}\n")
    print(recommendations[["primaryTitle", "startYear", "averageRating", "weighted_score"]].to_string(index=False))

# test all era options for Inception
for era in [None, "classic", "80s-90s", "2000s", "modern"]:
    print("=" * 55)
    get_recommendations_v3("Inception", era=era)


Because you watched: Inception
Era preference: any era

                                  primaryTitle  startYear  averageRating  weighted_score
                                  Interstellar       2014            8.7        8.676430
Star Wars: Episode V - The Empire Strikes Back       1980            8.7        8.660645
                    Terminator 2: Judgment Day       1991            8.6        8.557082
                            Back to the Future       1985            8.5        8.464640
                             Avengers: Endgame       2019            8.4        8.367239

Because you watched: Inception
Era preference: classic

         primaryTitle  startYear  averageRating  weighted_score
2001: A Space Odyssey       1968            8.3        8.245531
     The Great Escape       1963            8.2        8.071267
   Planet of the Apes       1968            8.0        7.868233
    The Wages of Fear       1953            8.1        7.775037
           Goldfinger       1964